In [1]:
pip install XlsxWriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 9.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import librosa
import numpy as np
import pandas as pd
import pywt
from scipy import stats
from math import sqrt

# ============================================================
# 1. Feature Extraction Functions
# ============================================================

def extract_mfcc(waveform, sr=16000, n_mfcc=13):
    mfccs = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=n_mfcc)
    return mfccs.mean(axis=1)

def extract_time_features(waveform):
    zcr = librosa.feature.zero_crossing_rate(waveform)[0].mean()
    rms = librosa.feature.rms(y=waveform)[0].mean()
    return np.array([zcr, rms])

def extract_wavelet(waveform):
    coeffs = pywt.wavedec(waveform, 'db4', level=4)
    feats = []
    for c in coeffs:
        feats.append(np.mean(c))
        feats.append(np.std(c))
    return np.array(feats[:10])   # 10 wavelet features

# ============================================================
# 2. Dataset Loader
# ============================================================

ROOT = "/kaggle/input/qmsat-dataset/ATS-data"

class_map = {"Music": "M", "Normal(Silence)": "NS", "SpiritualMeditation": "SM"}

data = []

for cls in class_map:
    folder = os.path.join(ROOT, cls)
    for file in os.listdir(folder):
        if file.endswith(".wav"):
            path = os.path.join(folder, file)

            wav, sr = librosa.load(path, sr=16000)

            mfcc = extract_mfcc(wav)
            time_f = extract_time_features(wav)
            wave = extract_wavelet(wav)

            feats = np.concatenate([mfcc, time_f, wave])

            data.append([class_map[cls]] + feats.tolist())

# Convert to DataFrame
feature_names = [f"MFCC_{i}" for i in range(13)] + ["ZCR", "RMS"] + [f"WV_{i}" for i in range(10)]
df = pd.DataFrame(data, columns=["Class"] + feature_names)

# ============================================================
# 3. Effect Sizes + Confidence Intervals
# ============================================================

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_sd = sqrt(((nx - 1)*np.var(x) + (ny - 1)*np.var(y)) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / pooled_sd

def eta_squared_anova(groups):
    grand_mean = np.mean(np.concatenate(groups))
    ss_between = sum(len(g)*(np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = sum(sum((g - grand_mean)**2) for g in groups)
    return ss_between / ss_total

def ci_95(data):
    mean = np.mean(data)
    se = stats.sem(data)
    h = se * stats.t.ppf(0.975, len(data)-1)
    return mean-h, mean+h

results = []

classes = df["Class"].unique()

for feat in feature_names:
    g1 = df[df["Class"] == "SM"][feat]
    g2 = df[df["Class"] == "M"][feat]
    g3 = df[df["Class"] == "NS"][feat]

    # ANOVA
    F, p = stats.f_oneway(g1, g2, g3)
    eta2 = eta_squared_anova([g1, g2, g3])

    # Pairwise Effect Sizes (Cohen's d)
    d_SM_M = cohens_d(g1, g2)
    d_SM_NS = cohens_d(g1, g3)
    d_M_NS = cohens_d(g2, g3)

    # Confidence intervals
    ciSM = ci_95(g1)
    ciM  = ci_95(g2)
    ciNS = ci_95(g3)

    results.append([
        feat, p, eta2,
        d_SM_M, d_SM_NS, d_M_NS,
        ciSM, ciM, ciNS
    ])

# Create results DataFrame
res_df = pd.DataFrame(results, columns=[
    "Feature", "ANOVA_p", "Eta2",
    "Cohen_d_SM_M", "Cohen_d_SM_NS", "Cohen_d_M_NS",
    "CI_SM", "CI_M", "CI_NS"
])

# ============================================================
# 4. SAVE ALL RESULTS TO EXCEL
# ============================================================

output_path = "/kaggle/working/ATS_effect_size_results.xlsx"
with pd.ExcelWriter(output_path) as writer:
    df.to_excel(writer, sheet_name="Raw Features", index=False)
    res_df.to_excel(writer, sheet_name="Effect Sizes + CIs", index=False)

print("Finished! Excel saved to:", output_path)


Finished! Excel saved to: /kaggle/working/ATS_effect_size_results.xlsx
